# 찍먹 · 새 음식 후보 집계 (1M_Final 레시피)
- **GPU 필요 없음** (런타임 기본 CPU 그대로)
- 위에서부터 차례로 실행. 전체 5~10분
- 결과는 `26-2 딥러닝/찍먹_실험기록/새음식후보/` 폴더에 CSV로 저장됨 → Claude가 읽어서 정리

궁금한 것
1. 레시피 중 사진 주소(`image1`)가 있는 비율
2. Food-101 101종이 레시피에 다 있는지 (레시피 수, 사진 있는 레시피 수)
3. Food-101에 없는 해외 음식 후보가 레시피에 얼마나 있는지
4. 레시피 제목에 자주 나오는 음식 이름 (데이터에서 직접 발견)
5. 사진 주소가 실제로 살아 있는지 샘플 확인

## 1. 드라이브 연결 · 레시피 불러오기

In [ ]:
import os, re, unicodedata
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
OUT = f"{find(TEAM, '찍먹_실험기록')}/새음식후보"
os.makedirs(OUT, exist_ok=True)

# 1.4GB라 필요한 칸만 읽음
df = pd.read_csv(f'{TEAM}/data1/1M_Final.csv', usecols=['title', 'image1'], encoding='utf-8', encoding_errors='replace')  # EDA와 같은 읽기 옵션
df['t'] = ' ' + df['title'].fillna('').str.lower().str.replace(r'[^a-z0-9 ]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True) + ' '
df['img'] = df['image1'].fillna('').str.startswith('http')
print(f'레시피 {len(df):,}개 | 사진 주소 있음 {df.img.sum():,}개 ({df.img.mean():.1%})')

## 2. 세는 함수
- 제목에 음식 이름(영어 표기들)이 단어로 들어간 레시피를 셈
- `제외후`: 이름 뒤에 salad·soup·dip 등 다른 요리가 붙은 경우를 뺀 수 (EDA 때 쓴 "다른 요리 빼기" 규칙의 간단 버전)

In [ ]:
OTHER = r'(?:salad|soup|dip|casserole|pizza|bites|bars|cupcakes?|sauce|seasoning|spice|dressing|wraps?|bowls?|pasta|inspired|style|flavored|popcorn|chips|cookies?)'

def count(aliases):
    names = '|'.join(re.escape(a) for a in aliases)
    hit = df['t'].str.contains(rf' (?:{names}) ', regex=True)
    other = df['t'].str.contains(rf' (?:{names}) {OTHER} ', regex=True)
    keep = hit & ~other
    return dict(레시피=int(hit.sum()), 제외후=int(keep.sum()), 제외후_사진있음=int((keep & df.img).sum()))

def status(n):
    return '표 가능(30+)' if n >= 30 else ('정보 부족(1~29)' if n > 0 else '없음 → 다른 데이터 필요')

## 3. Food-101 101종은 레시피에 다 있나?

In [ ]:
FOOD101 = '''apple_pie baby_back_ribs baklava beef_carpaccio beef_tartare beet_salad beignets bibimbap bread_pudding breakfast_burrito bruschetta caesar_salad cannoli caprese_salad carrot_cake ceviche cheesecake cheese_plate chicken_curry chicken_quesadilla chicken_wings chocolate_cake chocolate_mousse churros clam_chowder club_sandwich crab_cakes creme_brulee croque_madame cup_cakes deviled_eggs donuts dumplings edamame eggs_benedict escargots falafel filet_mignon fish_and_chips foie_gras french_fries french_onion_soup french_toast fried_calamari fried_rice frozen_yogurt garlic_bread gnocchi greek_salad grilled_cheese_sandwich grilled_salmon guacamole gyoza hamburger hot_and_sour_soup hot_dog huevos_rancheros hummus ice_cream lasagna lobster_bisque lobster_roll_sandwich macaroni_and_cheese macarons miso_soup mussels nachos omelette onion_rings oysters pad_thai paella pancakes panna_cotta peking_duck pho pizza pork_chop poutine prime_rib pulled_pork_sandwich ramen ravioli red_velvet_cake risotto samosa sashimi scallops seaweed_salad shrimp_and_grits spaghetti_bolognese spaghetti_carbonara spring_rolls steak strawberry_shortcake sushi tacos takoyaki tiramisu tuna_tartare waffles'''.split()

# 원래 파이프라인처럼 이름의 _ 를 공백으로 (단순 비교용이라 EDA 최종 수치와 조금 다를 수 있음)
f101 = pd.DataFrame([dict(음식=c, **count([c.replace('_', ' ')])) for c in FOOD101])
f101['상태'] = f101['제외후'].map(status)
f101.to_csv(f'{OUT}/food101_레시피수.csv', index=False, encoding='utf-8-sig')
print(f101['상태'].value_counts().to_string())
print('\n레시피가 적은 순 10개'); print(f101.sort_values('제외후').head(10).to_string(index=False))

## 4. Food-101에 없는 해외 음식 후보
해외여행에서 자주 보는 음식 위주로 골랐음. 필요하면 여기에 더 추가해서 다시 실행하면 됨.

In [ ]:
CAND = {
    # 한국
    '떡볶이': ['tteokbokki', 'ddeokbokki', 'topokki', 'spicy rice cake', 'spicy rice cakes'],
    '불고기': ['bulgogi'], '잡채': ['japchae', 'jap chae'], '김치찌개': ['kimchi jjigae', 'kimchi stew'],
    '김밥': ['kimbap', 'gimbap'], '삼겹살': ['samgyeopsal'], '한국식 프라이드치킨': ['korean fried chicken'],
    '파전': ['pajeon', 'korean pancake', 'scallion pancake', 'scallion pancakes'],
    # 일본
    '우동': ['udon'], '소바': ['soba'], '돈카츠': ['tonkatsu', 'katsu'], '오코노미야키': ['okonomiyaki'],
    '오니기리': ['onigiri', 'rice ball', 'rice balls'], '덴푸라': ['tempura'], '규동': ['gyudon', 'beef bowl'], '데리야키': ['teriyaki chicken', 'chicken teriyaki'],
    # 중국·대만·홍콩
    '마파두부': ['mapo tofu', 'ma po tofu'], '궁보계정': ['kung pao chicken', 'kung pao'], '샤오롱바오': ['xiaolongbao', 'soup dumplings', 'soup dumpling'],
    '차슈': ['char siu'], '콘지(죽)': ['congee', 'jook'], '탕수육/스위트앤사워': ['sweet and sour pork', 'sweet and sour chicken'],
    '딤섬/하가우': ['har gow', 'siu mai', 'shumai'], '버블티': ['bubble tea', 'boba tea', 'boba'],
    # 동남아
    '반미': ['banh mi'], '분짜': ['bun cha'], '똠얌': ['tom yum', 'tom yam'], '그린커리': ['green curry'], '마싸만커리': ['massaman curry', 'massaman'],
    '팟씨유': ['pad see ew'], '라브': ['larb', 'laab'], '쏨땀': ['som tam', 'papaya salad'], '카오소이': ['khao soi'],
    '락사': ['laksa'], '나시고렝': ['nasi goreng'], '렌당': ['rendang'], '사테': ['satay', 'sate'], '하이난 치킨라이스': ['hainanese chicken rice', 'chicken rice'],
    '룸피아': ['lumpia'], '아도보': ['adobo'],
    # 인도·중동
    '치킨 티카 마살라': ['tikka masala'], '버터치킨': ['butter chicken', 'murgh makhani'], '비리야니': ['biryani', 'biriyani'],
    '난': ['naan'], '달': ['dal', 'dhal', 'daal'], '팔락 파니르': ['palak paneer', 'saag paneer'],
    '샤와르마': ['shawarma'], '케밥': ['kebab', 'kabob', 'kebob'], '기로스': ['gyro', 'gyros'], '샥슈카': ['shakshuka'], '타진': ['tagine'], '쿠스쿠스': ['couscous'],
    # 유럽
    '무사카': ['moussaka'], '수블라키': ['souvlaki'], '슈니첼': ['schnitzel'], '굴라시': ['goulash'], '피에로기': ['pierogi', 'pierogies'],
    '라따뚜이': ['ratatouille'], '키슈': ['quiche'], '크레페': ['crepe', 'crepes'], '에그타르트(나타)': ['pastel de nata', 'egg tart', 'egg tarts'],
    '파니니': ['panini'], '젤라또': ['gelato'], '브라트부어스트': ['bratwurst'], '셰퍼드 파이': ['shepherd s pie', 'shepherds pie', 'cottage pie'],
    # 아메리카
    '부리또': ['burrito', 'burritos'], '엔칠라다': ['enchilada', 'enchiladas'], '타말레': ['tamale', 'tamales'], '엠파나다': ['empanada', 'empanadas'],
    '아레파': ['arepa', 'arepas'], '포케': ['poke bowl', 'poke'], '검보': ['gumbo'], '잠발라야': ['jambalaya'], '칠리': ['chili con carne'],
}
cand = pd.DataFrame([dict(음식=k, 검색어=', '.join(v), **count(v)) for k, v in CAND.items()])
cand['상태'] = cand['제외후'].map(status)
cand = cand.sort_values('제외후', ascending=False)
cand.to_csv(f'{OUT}/후보_레시피수.csv', index=False, encoding='utf-8-sig')
print(cand['상태'].value_counts().to_string())
print(); print(cand[['음식', '레시피', '제외후', '제외후_사진있음', '상태']].to_string(index=False))

## 5. 레시피 제목에 자주 나오는 두 단어 (데이터에서 직접 발견)
위 후보 목록에 없는 음식이 많이 있는지 보려는 것. 흔한 수식어(easy, best, homemade 등)는 뺌.

In [ ]:
from collections import Counter
STOP = set('a an and the with of in on for to my best easy quick homemade simple healthy recipe recipes style low fat free gluten vegan slow cooker instant pot crock mom s grandma grandmas classic perfect super']'.replace("]'", '').split())
known = {w for c in FOOD101 for w in c.split('_')}
bg = Counter()
for t in df['t']:
    w = [x for x in t.split() if x not in STOP and not x.isdigit()]
    bg.update(zip(w, w[1:]))
top = [(f'{a} {b}', n) for (a, b), n in bg.most_common(400) if not ({a, b} <= known)][:80]
disc = pd.DataFrame(top, columns=['두 단어', '레시피 수'])
disc.to_csv(f'{OUT}/제목_자주나오는_두단어.csv', index=False, encoding='utf-8-sig')
print(disc.head(40).to_string(index=False))

## 6. 사진 주소가 실제로 살아 있나 (샘플)
상위 후보 6개에서 각각 15개씩 열어 봄. 살아 있는 사진 몇 장은 화면에 작게 보여 줌 (품질 눈으로 확인용, 재배포 금지).

In [ ]:
import requests, io
from PIL import Image
from IPython.display import display

rows = []
for _, r in cand[cand['제외후_사진있음'] >= 30].head(6).iterrows():
    names = '|'.join(re.escape(a) for a in CAND[r['음식']])
    m = df['t'].str.contains(rf' (?:{names}) ', regex=True) & ~df['t'].str.contains(rf' (?:{names}) {OTHER} ', regex=True) & df.img
    urls = df.loc[m, 'image1'].sample(min(15, int(m.sum())), random_state=0)
    ok, thumbs = 0, []
    for u in urls:
        try:
            resp = requests.get(u, timeout=6, headers={'User-Agent': 'Mozilla/5.0'})
            if resp.ok and resp.headers.get('content-type', '').startswith('image'):
                ok += 1
                if len(thumbs) < 4:
                    im = Image.open(io.BytesIO(resp.content)).convert('RGB'); im.thumbnail((160, 160)); thumbs.append(im)
        except Exception:
            pass
    rows.append(dict(음식=r['음식'], 확인=len(urls), 살아있음=ok, 비율=round(ok / max(len(urls), 1), 2)))
    print(f"{r['음식']}: {ok}/{len(urls)} 살아 있음")
    if thumbs:
        canvas = Image.new('RGB', (170 * len(thumbs), 170), 'white')
        for i, im in enumerate(thumbs): canvas.paste(im, (170 * i, 0))
        display(canvas)
pd.DataFrame(rows).to_csv(f'{OUT}/사진주소_샘플확인.csv', index=False, encoding='utf-8-sig')

## 끝
다 돌았으면 Claude에게 "다 돌렸어"라고 말하면 `새음식후보/` 폴더의 CSV를 읽어서 정리해 줌.